# Solar Filament 2026 — final classical pipeline

This notebook reproduces the CPU-only radial-normalization and connected-component pipeline used for our strongest official submission (Kaggle submission ref `55113085`, public score `0.48`). The score is a platform result, not recomputed here.

## Reproducibility contract

Run from the repository root in a Kaggle notebook with the competition data attached. The notebook installs the pinned environment, imports only repository code, discovers the official test directory, generates `submission-classical.csv`, and fully audits it. For a local reproduction, set `SOLAR_IMAGE_DIR` and optionally `SOLAR_OUTPUT`.

In [ ]:
from pathlib import Path
import os, platform, subprocess, sys

REPO = Path.cwd()
assert (REPO / 'requirements.txt').exists(), 'Run this notebook from the repository root'
if os.environ.get('SOLAR_SKIP_INSTALL') != '1':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements.txt')], check=True)
else:
    print('Using preinstalled dependencies (SOLAR_SKIP_INSTALL=1)')
print({'python': sys.version, 'platform': platform.platform()})

In [ ]:
from solarfil.infer import infer_directory

override = os.environ.get('SOLAR_IMAGE_DIR')
candidates = [
    Path('/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images'),
    Path('/kaggle/input/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images'),
]
IMAGE_DIR = Path(override) if override else next((path for path in candidates if path.is_dir()), None)
if IMAGE_DIR is None:
    matches = list(Path('/kaggle/input').glob('**/test/test_images'))
    IMAGE_DIR = matches[0] if len(matches) == 1 else None
assert IMAGE_DIR is not None, 'Attach the official competition dataset'
images = sorted(IMAGE_DIR.glob('*.jpeg'))
assert images, f'No JPEG test images under {IMAGE_DIR}'
print({'test_directory': str(IMAGE_DIR), 'image_count': len(images)})

## Inference

Each image is radially normalized with robust annular statistics. Pixels in the dark disk-relative tail become candidates; four-connected components form instances; tiny components are removed and the remaining masks are encoded as compressed COCO RLE.

In [ ]:
OUTPUT = Path(os.environ.get('SOLAR_OUTPUT', '/kaggle/working/submission-classical.csv'))
row_count = infer_directory(IMAGE_DIR, OUTPUT)
print({'output': str(OUTPUT), 'prediction_rows': row_count})

In [ ]:
import csv
from scripts.audit_submission import audit
from solarfil.submission import decode_mask

with OUTPUT.open(newline='') as handle:
    reader = csv.DictReader(handle)
    assert reader.fieldnames == ['filament_id', 'segmentation_rle']
    submission = list(reader)
assert submission
identifiers = [row['filament_id'] for row in submission]
assert len(identifiers) == len(set(identifiers))
assert all(isinstance(row['segmentation_rle'], str) and row['segmentation_rle'] for row in submission)
known_ids = {path.stem for path in images}
submitted_ids = {value.rsplit('_', 1)[0] for value in identifiers}
assert submitted_ids <= known_ids
sample = submission[0]
sample_image = next(path for path in images if path.stem == sample['filament_id'].rsplit('_', 1)[0])
from PIL import Image
height, width = Image.open(sample_image).size[::-1]
decoded = decode_mask(sample['segmentation_rle'], (height, width))
assert decoded.shape == (height, width) and decoded.any()
receipt = audit(OUTPUT, IMAGE_DIR)
print({'rows': len(submission), 'images_with_predictions': len(submitted_ids), 'schema': 'valid', 'audit': receipt})
submission[:5]

## Submission and disclosure

Download `/kaggle/working/submission-classical.csv` and submit it through the official competition interface. The implementation is deterministic for a fixed dataset and dependency set. No pretrained weights, external datasets, or network inference services are used. AI-assisted coding was used to help prepare and review repository materials; the method, outputs, score claims, and final submission remain the entrant's responsibility.